<a href="https://colab.research.google.com/github/seungc1/Data_Analysis_Competition/blob/main/%EB%8D%B0%EC%9D%B4%ED%84%B0_%ED%95%84%ED%84%B0%EB%A7%81_%ED%9B%84_%ED%8C%8C%EC%9D%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

import warnings

warnings.filterwarnings(action='ignore')

In [ ]:
# (Colab) 시각화 한글폰트 설정을 위해 아래 코드를 실행하세요.
!apt -qq -y install fonts-nanum > /dev/null
!rm -rf ~/.cache/matplotlib

import matplotlib as mpl
import matplotlib.font_manager as fm
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
mpl.rcParams['font.family'] = 'NanumGothic'
mpl.rcParams['axes.unicode_minus'] = False

Cell 1: 환경 설정 및 공통 유틸리티

In [ ]:
# 1. 데이터별 메모리 최적화 타입 정의
DTYPE_CARD = {
    '기준연월': 'category', '가맹점행정구역분류시군구코드': 'category',
    '통합카드업종3레벨코드': 'category', '통합카드5세단위연령코드': 'category',
    '통합카드가구형태코드': 'category', '카드사용건수': 'int32', '카드사용금액': 'float32'
}

DTYPE_NICE = {
    '기준년월': 'category', '시군구코드(개정후)': 'category',
    '성별': 'category', '연령구간대': 'category', '직업구분': 'category',
    '분위코드(총대출잔액기준)': 'category', '대출잔액_보유대상자수': 'int32',
    '총대출잔액': 'float32', '월 연체보유자수 합계': 'int32'
}

# 2. 보안 규정 준수 필터링 함수
def apply_vdr_security(df, count_col):
    """건수가 5 미만인 로우를 삭제하여 보안 반출 규정 준수"""
    return df[df[count_col] >= 5].copy()

Cell 2: 대용량 데이터 최적화 로딩 엔진

In [ ]:
def process_large_txt(file_path, usecols, dtype, group_cols, agg_dict, count_col):
    """대용량 .txt 파일을 청크 단위로 처리하는 엔진"""
    chunk_list = []

    # 10만 행씩 청크 처리
    for chunk in pd.read_csv(file_path, sep='\t', usecols=usecols, dtype=dtype, chunksize=100000):
        # 집계 연산
        grouped = chunk.groupby(group_cols).agg(agg_dict)
        chunk_list.append(grouped)

    # 청크 결합 및 중간 집계
    master = pd.concat(chunk_list).groupby(level=group_cols).sum().reset_index()

    # 보안 마스킹 적용
    return apply_vdr_security(master, count_col)

Cell 3: 민간 데이터 일괄 통합 처리

In [ ]:
# 1. 카드 데이터 처리
CARD_COLS = ['기준연월', '가맹점행정구역분류시군구코드', '통합카드업종3레벨코드', '통합카드5세단위연령코드', '통합카드가구형태코드', '카드사용건수', '카드사용금액']
df_card = process_large_txt(
    '/Rdata1/r1_user138/dataset/P46_CARD_SALES/03.CARD_DOMESTIC_SEL/CARD_DOMESTIC_SEOUL_202401.txt',
    CARD_COLS, DTYPE_CARD, CARD_COLS[1:5], {'카드사용건수': 'sum', '카드사용금액': 'sum'}, '카드사용건수'
)

# 2. NICE 대출 데이터 처리
LOAN_COLS = ['기준년월', '시군구코드(개정후)', '성별', '연령구간대', '직업구분', '분위코드(총대출잔액기준)', '대출잔액_보유대상자수', '총대출잔액']
df_loan = process_large_txt(
    '/Rdata1/r1_user138/dataset/P47_NICE_CREDIT/01.LOAN_ADM/NICE_LOAN_AGE_202401.txt',
    LOAN_COLS, DTYPE_NICE, LOAN_COLS[1:6], {'대출잔액_보유대상자수': 'sum', '총대출잔액': 'sum'}, '대출잔액_보유대상자수'
)

# 3. NICE 소득 데이터 처리
INCOM_COLS = ['기준년월', '시군구코드(개정후)', '성별', '연령구간대', '직업구분', '분위코드(연소득기준)', '거주자수', '평균 연소득 금액']
df_income = process_large_txt(
    '/Rdata1/r1_user138/dataset/P47_NICE_CREDIT/02.INCOM_ADM/NICE_INCOM_AGE_202401.txt',
    INCOM_COLS, DTYPE_NICE, INCOM_COLS[1:6], {'거주자수': 'sum', '평균 연소득 금액': 'mean'}, '거주자수'
)

print("✅ 모든 민간/금융 데이터 통합 전처리 완료.")